# The “Memory” of LLMs: Using RoPE Extrapolation as an Entry Point

When people talk about the “memory” of large language models, they usually distinguish among three types: **parametric memory** stored in the model parameters, **contextual memory** supported by attention, and **external memory** represented by RAG, databases, and tool use. Correspondingly, people often evaluate a model’s long-context ability using tasks such as LongBench, RULER, and Needle-in-a-Haystack.

However, these evaluations are mostly framed from the perspective of humans using a tool: Can the model answer questions? Can it find evidence? Can it complete long-document tasks? They measure behavioral usefulness, rather than what kind of contextual ability a Transformer has at the mechanistic level.

If we start from a lower-level mechanism, the question is no longer how many characters a Transformer can “remember,” but rather:

> How does an attention network with finite depth, finite width, and positional encoding preserve, route, combine, and use information from the context as the sequence length increases?

Note: Here, “routing” is not a strict mathematical term from the original Transformer paper. It is a descriptive term often used in recent analyses of internal Transformer mechanisms. It refers to how the information of a certain token in the input passes through attention, residual connections, MLPs, and multi-layer representation transformations, and eventually influences the output. In other words, routing asks which path information takes inside the network.

### A Layered Breakdown of Contextual Ability

From this perspective, the contextual ability of a Transformer can be decomposed into at least the following levels.

First, **attention reachability**. Can the information of a certain token be accessed by later tokens? The core object here is the attention matrix. Metrics such as attention entropy, attention mass, and attention distance distribution can be used to observe whether the model actually establishes effective connections between distant positions.

Second, **information retention**. After previous information passes through layer-by-layer transformations, how much of it remains in the hidden states? This can be analyzed through mutual information, linear probing, representation similarity, and related methods. If distant information is formally still inside the context window but is no longer recoverable in the representation space, then the model has effectively “forgotten” it.

Third, **positional-encoding extrapolation**. Can the model really handle contexts longer than the training length? This involves the properties of positional encoding mechanisms such as RoPE, ALiBi, and positional interpolation. For RoPE, the key question is not how high the model scores on a long-document QA benchmark, but what happens to rotation phases, attention distributions, and representation structures when the relative distance exceeds the training window.

Fourth, **signal-to-noise ratio**. In a long context, will the information of the target token be diluted by a large number of irrelevant tokens? As the context length $n$ increases, the competing terms in softmax normalization also increase. Even if the target information still exists, its effective weight in the attention distribution may decrease.

Fifth, **compositional complexity**. Can the model implement multi-step information transmission through multi-layer attention? A single attention layer is more like one global read, whereas multi-layer attention can form path compositions across tokens and layers. Long-context ability depends not only on whether distant information can be read, but also on whether distributed information can be combined into a new representation.

Sixth, **effective context length**. This is different from the model’s nominal context window. The truly effective context length should refer to the maximum distance at which a distant information source still has a measurable influence on the output distribution. If deleting a distant token barely changes the output distribution, then that token is inside the window but is not actually being used.

The strong performance of Transformers easily gives the impression that as long as all information is placed into the input, attention will automatically collect and use it, achieving something like full memory. But this is more of a behavioral intuition, not a mechanistic fact. In practice, even when a rule is repeatedly emphasized in the prompt, the model may still violate it later during generation. However, this is not necessarily just a problem of “poor memory,” and LLMs should not be understood through the psychology of training subordinates or reminding humans.

Among these factors, positional encoding is especially important. Attention itself only computes correlations between token representations, while information retrieval in long contexts first requires the model to correctly judge relative positional relationships between tokens. If positional relationships become distorted as length grows, then distant tokens may still be inside the window, but they may no longer participate in attention computation in a way that is familiar from training. In other words, being able to “remember” first requires the model to maintain a stable positional geometry on longer sequences.

Therefore, this article will **use RoPE as an entry point to examine what happens when a language model trained on fixed-length token blocks encounters a context longer than its training window**. The focus includes visualizing RoPE’s numerical extrapolation behavior, changes in attention under long-distance positional relationships, and the strength with which contextual token information is retained. Through this, we try to reinterpret the so-called “memory” of large language models from a mechanistic perspective.

In summary, long-context ability can be divided into three levels:

**Behavior-level metrics**: LongBench, RULER, Needle-in-a-Haystack.  
**Mechanism-level metrics**: attention entropy, attention distance, representation retention, causal ablation, information flow.  
**Mathematical-level questions**: complexity, capacity, positional extrapolation, softmax dilution, depth, and compositional paths.

This article is not concerned with whether a model is “useful” on long-context tasks, but with how a Transformer actually preserves, transmits, and uses information under length extrapolation.


## The Principle and Implementation of RoPE

Common positional encoding methods include absolute positional encoding, relative positional encoding, ALiBi, and Rotary Position Embedding (RoPE), which is currently used by many large language models.

Compared with traditional positional encodings, the most important feature of RoPE is that it does not directly encode absolute positions. Instead, it uses rotation transformations so that attention computation naturally depends on the relative distance between tokens. This property gives RoPE better generalization ability in long-context tasks and makes it an important entry point for studying Transformer contextual ability and length extrapolation.

Therefore, when discussing the “memory” of large language models, we first need to understand how RoPE works, why it can support long contexts, and why it gradually fails after exceeding the training length.


### The Mathematical Principle of RoPE

The core idea of RoPE can be summarized in one sentence:

> **By applying position-dependent rotation transformations to Query and Key, the inner product in Attention depends only on the relative position between tokens, rather than on their absolute positions.**

Unlike traditional positional encoding, which directly adds a position vector to the embedding, RoPE incorporates positional encoding into the Attention computation process.


#### What RoPE Operates On

Let the input sequence length be $L$, and let the model hidden dimension be $d_{\mathrm{model}}$. After embedding and several preceding Transformer layers, the input representation of the current layer is

$$
X \in \mathbb{R}^{L \times d_{\mathrm{model}}}.
$$

For a certain attention head, Query and Key are obtained through linear projections:

$$
Q = XW_Q,
\qquad
K = XW_K.
$$

where

$$
W_Q,W_K \in \mathbb{R}^{d_{\mathrm{model}} \times d},
$$

and $d$ is the dimension of each attention head. Therefore,

$$
Q,K \in \mathbb{R}^{L \times d}.
$$

At this point, the **$m$-th row** of $Q$ represents the Query vector corresponding to the $m$-th token:

$$
q_m = Q[m,:] \in \mathbb{R}^{d}.
$$

The **$n$-th row** of $K$ represents the Key vector corresponding to the $n$-th token:

$$
k_n = K[n,:] \in \mathbb{R}^{d}.
$$

That is, $q_m$ is not an independent Query matrix, but a row vector in the full Query matrix $Q$ corresponding to position $m$. Similarly, $k_n$ is a row vector in the Key matrix $K$ corresponding to position $n$. Here, $d$ refers to the feature dimension inside each attention head. We assume by default that the head dimension $d$ is even. If $d$ is odd, RoPE can only be applied to an even number of dimensions, leaving the remaining dimension unchanged, or the model can simply be designed so that $d$ is even.


#### Pairwise Splitting

Next, RoPE **splits the feature dimensions inside the same Query or Key vector into multiple two-dimensional subspaces by grouping adjacent dimensions**. This does not mean grouping different tokens in pairs, nor does it mean grouping Query and Key in pairs.

For example, for the Query vector

$$
q_m=(q_{m,0},q_{m,1},q_{m,2},q_{m,3},\cdots,q_{m,d-2},q_{m,d-1}),
$$

RoPE divides it into

$$
(q_{m,0},q_{m,1}),\quad
(q_{m,2},q_{m,3}),\quad
\cdots,\quad
(q_{m,d-2},q_{m,d-1}).
$$

For the Key vector

$$
k_n=(k_{n,0},k_{n,1},k_{n,2},k_{n,3},\cdots,k_{n,d-2},k_{n,d-1}),
$$

the same split is applied:

$$
(k_{n,0},k_{n,1}),\quad
(k_{n,2},k_{n,3}),\quad
\cdots,\quad
(k_{n,d-2},k_{n,d-1}).
$$

In other words, RoPE rotates both Query and Key. The only difference is that Query uses its own position $m$, while Key uses its own position $n$.

For the $i$-th two-dimensional subspace, define the angular frequency for token positions as

$$
\theta_i = 10000^{-2i/d},
\qquad i=0,1,\cdots,\frac d2-1.
$$

At position $m$, the rotation matrix corresponding to the $i$-th two-dimensional subspace is

$$
R_i(m)=
\begin{bmatrix}
\cos(m\theta_i) & -\sin(m\theta_i)\\
\sin(m\theta_i) & \cos(m\theta_i)
\end{bmatrix}.
$$

Therefore, the $i$-th two-dimensional subspace in the Query vector is rotated into

$$
\begin{bmatrix}
\tilde q_{m,2i}\\
\tilde q_{m,2i+1}
\end{bmatrix}
=
R_i(m)
\begin{bmatrix}
q_{m,2i}\\
q_{m,2i+1}
\end{bmatrix}.
$$

Similarly, the $i$-th two-dimensional subspace in the Key vector is rotated into

$$
\begin{bmatrix}
\tilde k_{n,2i}\\
\tilde k_{n,2i+1}
\end{bmatrix}
=
R_i(n)
\begin{bmatrix}
k_{n,2i}\\
k_{n,2i+1}
\end{bmatrix}.
$$

Combining all two-dimensional subspaces gives a block diagonal matrix:

$$
\mathcal R(m)
=
\operatorname{diag}
\left(
R_0(m),
R_1(m),
\cdots,
R_{d/2-1}(m)
\right).
$$

Thus, the Query and Key after RoPE can be written as

$$
\tilde q_m = \mathcal R(m)q_m,
\qquad
\tilde k_n = \mathcal R(n)k_n.
$$


#### Applying RoPE

After $q_m$ and $k_n$ are processed by RoPE, the Attention score no longer uses the original inner product

$$
q_m^\top k_n,
$$

but instead uses the rotated inner product

$$
\tilde q_m^\top \tilde k_n.
$$

Expanding it gives

$$
\begin{aligned}
\tilde q_m^\top \tilde k_n
&=
\left(\mathcal R(m)q_m\right)^\top
\left(\mathcal R(n)k_n\right)\\
&=
q_m^\top \mathcal R(m)^\top \mathcal R(n) k_n.
\end{aligned}
$$

Because a two-dimensional rotation matrix satisfies

$$
R_i(m)^\top R_i(n)=R_i(n-m),
$$

the full block diagonal rotation matrix also satisfies

$$
\mathcal R(m)^\top \mathcal R(n)
=
\mathcal R(n-m).
$$

Therefore,

$$
\tilde q_m^\top \tilde k_n
=
q_m^\top \mathcal R(n-m) k_n.
$$

Correspondingly, after RoPE is added, the attention score can be written as

$$
\tilde s_{mn}
=
\frac{\tilde q_m^\top \tilde k_n}{\sqrt{d}}.
$$

Substituting the result above gives

$$
\tilde s_{mn}
=
\frac{
q_m^\top \mathcal R(n-m) k_n
}{\sqrt{d}}.
$$

Here, $\tilde s_{mn}$ denotes the attention score between the Query at position $m$ and the Key at position $n$, and $d$ is the dimension of each attention head. Finally, note that RoPE is usually applied only to Query and Key, not to Value. Therefore, RoPE changes the weight computation that determines “where to look,” rather than directly rotating the retrieved content $v_n$.


#### The Key Point

From the derivation above, we can see the most important property of RoPE: before Query and Key enter the attention score, they do indeed use their own absolute positions $m$ and $n$. The Query at position $m$ is rotated by $m\theta_i$, and the Key at position $n$ is rotated by $n\theta_i$.

However, when the two vectors take an inner product, what actually matters is not the two absolute angles themselves, but the angular difference between them:

$$
n\theta_i - m\theta_i = (n-m)\theta_i.
$$

That is, for the $i$-th two-dimensional subspace, what the attention score sees is not “the Query is at position $m$ and the Key is at position $n$,” but rather:

$$
\text{The Key is offset by } n-m \text{ tokens relative to the Query.}
$$

For example, if the current position is $m=100$ and the attended token is at $n=90$, then the positional information left by RoPE in the inner product is

$$
n-m = -10.
$$

If the current position is changed to $m=1000$ and the attended token is at $n=990$, then

$$
n-m = -10.
$$

The two cases have completely different absolute positions: one occurs around position 100, and the other around position 1000. But from the perspective of RoPE’s attention score, they have the same relative-distance structure: in both cases, the Key is 10 tokens before the Query.

Therefore, RoPE does not directly tell the model “this is the 100th token” or “this is the 1000th token.” Instead, it first uses absolute positions to determine how much Query and Key should each be rotated. Then, when the two vectors take an inner product, the two absolute rotation angles cancel each other out, leaving only the phase difference caused by relative displacement.

So, more precisely:

> RoPE uses absolute positions to rotate Query and Key, but the attention score ultimately senses the relative positional difference between Query and Key.

From this perspective, the essence of RoPE is:

> Introducing position-dependent rotations into the feature space of Query and Key, so that the relative positions between tokens are naturally incorporated into the Attention score.

This is also one reason why RoPE is more suitable for long contexts than ordinary absolute positional encoding: the model does not need to learn a separate position representation for every absolute position. Instead, it can learn relative-distance patterns such as “1 token apart,” “10 tokens apart,” or “100 tokens apart.”

It should be noted that RoPE itself does not explicitly output a variable called “relative distance.” It only encodes relative positional information into the inner-product structure of Query and Key through rotation phase differences. During training, the Transformer learns by itself how to use this structure: some attention heads may learn to attend to nearby tokens, while others may learn to attend to sentence beginnings, paragraph boundaries, or longer-distance dependencies. In other words, relative positional information is not directly read out by a handcrafted rule; rather, it exists as a usable geometric structure inside the attention score, and the model learns to parse and use it under the language-modeling objective.

Therefore, more accurately, RoPE provides a “relative-position-resolvable representation space”: it encodes relative distance into inner-product relationships, and the Transformer learns through training to extract useful positional patterns from these changes in inner products.


#### Advantages of RoPE

This design brings two important advantages.

First, because positional relationships are expressed through relative distance rather than fixed position indices, the model has better length generalization than absolute positional encoding to some extent.

Second, different dimensions correspond to different rotational angular frequencies. Low-frequency dimensions change slowly and can stably represent long-distance positional relationships; high-frequency dimensions change more quickly and can provide fine-grained local positional information. Multiple frequencies together form a representation similar to a Fourier basis, enabling the model to perceive both local and global positional relationships.

However, this representation also plants the seeds of length-extrapolation failure. Because the rotation angle grows linearly with position, high-frequency components begin to oscillate sharply when the context length greatly exceeds the training window, while low-frequency components also gradually accumulate mismatch. Although RoPE can mathematically compute rotation matrices for arbitrary lengths, what the model actually learned is only the phase distribution inside the training window. Therefore, as inference length keeps increasing, the positional relationships in Attention gradually deviate from the distribution learned during training. This is an important reason for long-context performance degradation and will be the focus of the later analysis.


### Why Not Directly Encode Relative Positions?

Since RoPE ultimately aims to make the attention score perceive the relative distance $n-m$, a natural question is: why not directly encode relative positions?

In fact, directly encoding relative positions is certainly possible. Many positional encoding methods do exactly this. For example, one can directly add a relative position bias to the attention score:

$$
s_{mn}
=
\frac{q_m^\top k_n}{\sqrt{d}}
+
b_{n-m}.
$$

Here, $s_{mn}$ denotes the attention score between the Query at position $m$ and the Key at position $n$, and $b_{n-m}$ is a position bias determined only by the relative distance $n-m$.

This method is very intuitive. It is equivalent to directly telling the model:

> The current position $m$ and the attended position $n$ differ by $n-m$ tokens.

For example, if $n-m=-10$, then the position bias for distance $-10$, namely $b_{-10}$, is used; if $n-m=-100$, then the position bias for distance $-100$, namely $b_{-100}$, is used.

However, this type of direct relative positional encoding usually has several problems.

#### First, it is often **content-independent**

In the following formula,

$$
s_{mn}
=
\frac{q_m^\top k_n}{\sqrt{d}}
+
b_{n-m},
$$

the relative position term $b_{n-m}$ depends only on distance, not on the content of the tokens. That is, whether the current position is a noun, a verb, a bracket, or a variable name, as long as the relative distance is the same, the added bias is the same.

This is certainly useful, but it is more like adding a fixed positional prior to the attention score. For example, the model can learn that “nearby tokens are usually more important,” or that “tokens at certain distances are more worth attending to.” However, this relative position term does not change how Query and Key match each other in terms of content.

RoPE works differently. After RoPE is added, the attention score can be written as

$$
\tilde s_{mn}
=
\frac{
q_m^\top \mathcal R(n-m) k_n
}{\sqrt{d}}.
$$

Here, the relative distance $n-m$ is not added to the attention score as an extra bias; instead, it directly participates in the inner-product computation between Query and Key.

If relative position bias is used, the attention score is

$$
s_{mn}
=
\frac{q_m^\top k_n}{\sqrt d}
+
b_{n-m}.
$$

This means: first compute the content similarity between $q_m$ and $k_n$, and then add an extra score according to the distance $n-m$. This positional term $b_{n-m}$ is independent of the specific content of Query and Key. As long as the relative distance is the same, the added score is the same.

In other words, the role of the bias is more like this:

> No matter what the two tokens contain, as long as they are separated by this distance, uniformly add or subtract some score.

RoPE works differently. The attention score after RoPE is

$$
\tilde s_{mn}
=
\frac{
q_m^\top \mathcal R(n-m) k_n
}{\sqrt d}.
$$

Here, $\mathcal R(n-m)$ first rotates the representation of Key according to the relative distance, and then the rotated Key takes an inner product with Query. That is, the distance $n-m$ does not change an extra score; it changes the alignment between $q_m$ and $k_n$ in vector space.

We can examine only one two-dimensional subspace. Let

$$
q=(a,b),
\qquad
k=(c,d),
$$

and let the rotation angle corresponding to the relative distance be

$$
\phi=(n-m)\theta_i.
$$

Then the inner product of this pair of two-dimensional vectors under RoPE is

$$
q^\top R(\phi)k
=
(ac+bd)\cos\phi
+
(bc-ad)\sin\phi.
$$

We can see that the positional relationship does not simply contribute a fixed bias $b_{n-m}$. The relative distance $\phi$ changes how different content components are combined through $\cos\phi$ and $\sin\phi$. Therefore, RoPE does not make the position itself “contentful”; rather, it lets positional relationships enter the content-matching process in a multiplicative form. It places “whether the two tokens match in content” and “how far apart they are” into the same inner-product structure for joint computation.

In other words, even for the same distance $-10$, different Queries and Keys can be affected differently; the same relative distance $n-m$ can have different effects on different $q_m$ and $k_n$; and the same pair of $q_m$ and $k_n$ can produce different inner-product results under different relative distances.

A potential advantage of this design is that **the model can learn during training how to place different types of information into different frequency subspaces**. Some heads can use high-frequency dimensions to handle local dependencies, while others can use low-frequency dimensions to handle longer-distance relationships. RoPE provides a positional geometric structure that the model can use, rather than directly specifying how much score should be added for a particular distance.

#### Second, directly learning a relative position table exposes the length-extrapolation problem more clearly.

Suppose we learn a position bias for every relative distance:

$$
b_{-1}, b_{-2}, b_{-3}, \cdots, b_{-L}.
$$

If the training window length is $L$, then the model only learns distance parameters within this range. Once the context length during inference exceeds the training window, distances without corresponding parameters from training will appear, for example:

$$
b_{-10000},\quad b_{-20000},\quad b_{-30000}.
$$

At this point, we must additionally specify how these unseen distances should be handled, such as truncation, interpolation, bucketing, or making distant positions share the same bias. In other words, a directly learned relative position table does not naturally define positional relationships outside the training range.

RoPE differs in that it does not store a separate parameter for every relative distance. Instead, it uses continuous functions to generate positional relationships:

$$
\cos((n-m)\theta_i),
\qquad
\sin((n-m)\theta_i).
$$

Therefore, at the representation level, any relative distance $n-m$ can be substituted into the computation. Even if $n-m$ exceeds the maximum distance seen during training, RoPE can still give a definite rotation phase.

But this does not mean that RoPE truly solves the length-extrapolation problem. It only guarantees that “unseen distances are mathematically computable”; it does not guarantee that the model has “learned how to use these unseen distances.”

These are two different levels of the problem:

- **Representation level**: Are positional relationships beyond the training length defined?
- **Learning level**: Has the model learned during training how to use these positional relationships?

At the representation level, RoPE is more natural than pure lookup-table relative positional encoding, because it does not need to learn a separate parameter for every distance, nor does it immediately encounter the problem of “no such position parameter” after exceeding the training length. But at the learning level, RoPE still faces the same out-of-distribution extrapolation problem: during training, the model has only seen phase combinations within a finite range; during inference, it encounters phase patterns at longer distances, which it still may not be able to interpret stably.

Therefore, the advantage of RoPE should not be understood as “it allows the model to automatically learn dependencies of arbitrary length.” More accurately, RoPE provides a continuous, parameter-shared positional geometry that makes length extrapolation formally possible; but whether this structure can actually be used by the model still depends on the training length, frequency design, model scale, and whether the attention heads have learned extrapolatable positional rules.

#### Third, RoPE is implementation-friendly for inference in decoder-only language models.

In autoregressive generation, the model caches the Keys and Values of past tokens, namely the KV cache. When using RoPE, the Key at a past position $n$ can be rotated during generation and then cached:

$$
\tilde k_n = \mathcal R(n)k_n.
$$

When the model generates at a new position $m$, it only needs to compute the rotation of the current Query:

$$
\tilde q_m = \mathcal R(m)q_m.
$$

Then it can directly compute

$$
\tilde q_m^\top \tilde k_n.
$$

In this way, RoPE is highly compatible with the matrix multiplication form of ordinary attention and does not require explicitly constructing a complex relative position table at every step.
